In [2]:
import sys
!{sys.executable} -m ensurepip --upgrade

Looking in links: /var/folders/dd/tpdzgsps29l6xqx8hm7z2_rr0000gp/T/tmpwhx5cp1x


In [3]:
# Install dependencies
%pip install anthropic python-dotenv


[notice] A new release of pip is available: 25.1.1 -> 26.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
# Load env vars

from dotenv import load_dotenv

load_dotenv()

True

In [5]:
# Create an API client

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [7]:
from anthropic.types import MessageParam
from collections.abc import Iterable

def add_user_message(messages: list[MessageParam], text: str):
    user_message: MessageParam = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages: list[MessageParam], text: str):
    assistant_message: MessageParam = {"role": "assistant", "content": text }
    messages.append(assistant_message)

from anthropic.types import TextBlock, Message

def get_message_text(message: Message):
    return next(
        (block.text for block in message.content if isinstance(block, TextBlock)),
        "" # empty string if none
    )

def chat(messages: Iterable[MessageParam], system: str|None = None, temperature = 1.0, stop_sequences: list[str] = []):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return get_message_text(message)

In [9]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "Here goes the tasks json:\n```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [ ]:
dataset = generate_dataset()

with open("data/007_generate_eval_dataset-tasks.json", "w") as f:
    json.dump(dataset, f, indent=2)